In [ ]:
# %pip install -U qbraid qiskit numpy==2.4 pandas statsmodels scikit-learn tqdm

In [ ]:
# %%
from pathlib import Path
import json

import numpy as np
import pandas as pd
import statsmodels.api as sm
from qbraid import QbraidProvider
from sklearn.metrics import mean_absolute_error, mean_squared_error
from QRC_model_qbraid import QRC_Model_QBraid

MODE = "subset"  # "smoke" for real job submission or "subset" for local run

DEVICE_ID = "aws:rigetti:qpu:cepheus-1-108q"
DATASET_PATH = Path("final_qrc_dataset.csv")
TICKER = None

NUM_QUBITS = 5
N_TROTTER = 2
DT = 0.1
FEEDBACKS = (0.1,)
SHOTS = 500
SEED = 0
RIDGE_PARAM = 1.0e-6
N_WASHOUT = 2

TRAIN_STEPS = 8
TEST_STEPS = 4

CREDIT_BUDGET = 900.0
PER_TASK_CREDIT = 30.0
PER_SHOT_CREDIT = 0.043

PRINT_JOB_IDS = True
OUTPUT_DIR = Path("qbraid_results")

# Safety lock
CONFIRM_REMOTE_RUN = False

In [ ]:
# %%
def print_device_metadata(device):
    print(f"Selected qBraid device: {device}")
    metadata_method = getattr(device, "metadata", None)

    if callable(metadata_method):
        try:
            print("Device metadata:")
            print(metadata_method())
        except Exception as exc:
            print(f"Could not retrieve device metadata: {exc}")


def estimate_and_print_cost(
    tasks,
    shots,
    per_task_credit,
    per_shot_credit,
    credit_budget,
):
    credits_per_task = per_task_credit + shots * per_shot_credit
    estimated_total = tasks * credits_per_task

    print("Remote execution estimate")
    print("-------------------------")
    print(f"Tasks:                {tasks}")
    print(f"Shots per task:       {shots}")
    print(f"Credits per task:     {credits_per_task:,.3f}")
    print(f"Estimated credits:    {estimated_total:,.3f}")
    print(f"Configured budget:    {credit_budget:,.3f}")
    print(f"Estimated remaining:  {credit_budget - estimated_total:,.3f}")

    return estimated_total

# %%
def build_har_dataframe(csv_path):
    csv_path = Path(csv_path)

    if not csv_path.exists():
        raise FileNotFoundError(f"Dataset not found: {csv_path.resolve()}")

    data = pd.read_csv(csv_path)

    required_columns = {"ticker", "date", "y", "split"}
    missing = required_columns.difference(data.columns)

    if missing:
        raise ValueError(
            "Dataset is missing required columns: "
            + ", ".join(sorted(missing))
        )

    data["date"] = pd.to_datetime(data["date"], errors="raise")
    data = data.sort_values(["ticker", "date"]).copy()

    grouped_y = data.groupby("ticker")["y"]

    data["y_d"] = grouped_y.shift(1)
    data["y_w"] = data.groupby("ticker")["y"].transform(
        lambda s: s.rolling(window=5).mean().shift(1)
    )
    data["y_m"] = data.groupby("ticker")["y"].transform(
        lambda s: s.rolling(window=22).mean().shift(1)
    )
    data["y_next"] = grouped_y.shift(-1)

    data["_next_split"] = data.groupby("ticker")["split"].shift(-1)
    data = data[data["_next_split"] == data["split"]].copy()

    data = data.dropna(
        subset=["y_d", "y_w", "y_m", "y_next", "split"]
    ).copy()

    train = data[data["split"] == "train"].copy()

    if train.empty:
        raise ValueError("No training rows remain after preprocessing.")

    X_train = sm.add_constant(
        train[["y_d", "y_w", "y_m"]],
        has_constant="add",
    )
    har_model = sm.OLS(train["y_next"], X_train).fit()

    X_all = sm.add_constant(
        data[["y_d", "y_w", "y_m"]],
        has_constant="add",
    )
    data["y_hat_HAR"] = har_model.predict(X_all)

    return data, har_model


def choose_ticker(data, requested=None):
    train_tickers = set(
        data.loc[data["split"] == "train", "ticker"].astype(str)
    )
    test_tickers = set(
        data.loc[data["split"] == "test", "ticker"].astype(str)
    )

    common = sorted(train_tickers.intersection(test_tickers))

    if not common:
        raise ValueError("No ticker is present in both splits.")

    if requested is not None:
        requested = str(requested)

        if requested not in common:
            raise ValueError(
                f"Ticker {requested!r} is unavailable. "
                f"Available choices: {common}"
            )

        return requested

    return common[0]


def build_single_ticker_arrays(
    data,
    ticker,
    train_steps,
    test_steps,
):
    ticker_data = data[
        data["ticker"].astype(str) == str(ticker)
    ].copy()

    train = (
        ticker_data[ticker_data["split"] == "train"]
        .sort_values("date")
        .tail(train_steps)
    )

    test = (
        ticker_data[ticker_data["split"] == "test"]
        .sort_values("date")
        .head(test_steps)
    )

    if len(train) < train_steps:
        raise ValueError(
            f"Ticker {ticker!r} has only {len(train)} training rows."
        )

    if len(test) < test_steps:
        raise ValueError(
            f"Ticker {ticker!r} has only {len(test)} testing rows."
        )

    x_train = train["y_next"].to_numpy(dtype=float)[None, :]
    yhar_train = train["y_hat_HAR"].to_numpy(dtype=float)[None, :]

    x_test = test["y_next"].to_numpy(dtype=float)[None, :]
    yhar_test = test["y_hat_HAR"].to_numpy(dtype=float)[None, :]
    test_dates = test["date"].to_numpy()

    return x_train, yhar_train, x_test, yhar_test, test_dates

# %%
def evaluate(y_true_log, y_pred_log, label):
    mse = mean_squared_error(y_true_log, y_pred_log)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true_log, y_pred_log)

    rv_true = np.exp(y_true_log)
    rv_pred = np.exp(y_pred_log)
    ratio = rv_true / rv_pred
    qlike = float(np.mean(ratio - np.log(ratio) - 1.0))

    metrics = {
        "mse": float(mse),
        "rmse": rmse,
        "mae": float(mae),
        "qlike": qlike,
    }

    print(
        f"{label:>12s}  "
        f"MSE: {metrics['mse']:.4f}  "
        f"RMSE: {metrics['rmse']:.4f}  "
        f"MAE: {metrics['mae']:.4f}  "
        f"QLIKE: {metrics['qlike']:.5f}"
    )

    return metrics


def create_qrc_model(device, max_tasks):
    return QRC_Model_QBraid(
        num_qubits=NUM_QUBITS,
        qbraid_device=device,
        ridge_param=RIDGE_PARAM,
        f_bs=FEEDBACKS,
        dt=DT,
        n_trotter=N_TROTTER,
        shots=SHOTS,
        seed=SEED,
        n_washout=N_WASHOUT,
        max_tasks=max_tasks,
        print_job_ids=PRINT_JOB_IDS,
    )

In [ ]:
# %%
provider = QbraidProvider()
device = provider.get_device(DEVICE_ID)
print_device_metadata(device)

# %% [markdown]
# ## Estimate task and credit usage

# %%
if MODE == "smoke":
    REQUIRED_TASKS = len(FEEDBACKS)

elif MODE == "subset":
    REQUIRED_TASKS = QRC_Model_QBraid.estimate_required_tasks(
        train_steps=TRAIN_STEPS,
        test_steps=TEST_STEPS,
        n_reservoirs=len(FEEDBACKS),
        n_tickers=1,
        n_washout=N_WASHOUT,
    )

else:
    raise ValueError("MODE must be 'smoke' or 'subset'.")

ESTIMATED_CREDITS = estimate_and_print_cost(
    tasks=REQUIRED_TASKS,
    shots=SHOTS,
    per_task_credit=PER_TASK_CREDIT,
    per_shot_credit=PER_SHOT_CREDIT,
    credit_budget=CREDIT_BUDGET,
)

if ESTIMATED_CREDITS > CREDIT_BUDGET:
    raise RuntimeError(
        "Estimated cost exceeds CREDIT_BUDGET."
    )

# %% [markdown]
# ## Optional one-task smoke test

# %%
if MODE == "smoke":
    smoke_model = create_qrc_model(
        device=device,
        max_tasks=REQUIRED_TASKS,
    )

    smoke_features = smoke_model.evolve_qrc(t0=0.1)

    print("Smoke test completed.")
    print(f"Feature length: {len(smoke_features)}")
    print(f"Features: {np.asarray(smoke_features)}")
    print(f"Job IDs: {smoke_model.job_ids}")

else:
    print("Skipped because MODE is not 'smoke'.")

# %% [markdown]
# ## Prepare bounded dataset subset
# 
# This does not submit quantum jobs.

# %%
if MODE == "subset":
    qrc_data, har_model = build_har_dataframe(DATASET_PATH)
    selected_ticker = choose_ticker(qrc_data, TICKER)

    (
        x_train,
        yhar_train,
        x_test,
        yhar_test,
        test_dates,
    ) = build_single_ticker_arrays(
        data=qrc_data,
        ticker=selected_ticker,
        train_steps=TRAIN_STEPS,
        test_steps=TEST_STEPS,
    )

    print(f"Selected ticker: {selected_ticker}")
    print(f"x_train shape:   {x_train.shape}")
    print(f"x_test shape:    {x_test.shape}")

else:
    print("Skipped because MODE is not 'subset'.")

# %% [markdown]
# ## Submit subset experiment
# 
# Remote tasks are submitted only when:
# 
# ```python
# MODE == "subset"
# CONFIRM_REMOTE_RUN is True
# ```

# %%
if MODE != "subset":
    print("Skipped because MODE is not 'subset'.")

elif not CONFIRM_REMOTE_RUN:
    print(
        "No remote subset tasks submitted. "
        "Set CONFIRM_REMOTE_RUN = True after checking the estimate."
    )

else:
    qrc_model = create_qrc_model(
        device=device,
        max_tasks=REQUIRED_TASKS,
    )

    qrc_model.train(x_train, yhar_train)
    qrc_model.fit()

    qrc_predictions = qrc_model.forward_one_shot(
        x_test,
        yhar_test,
    )

    y_true = x_test[0, 1:]
    y_har = yhar_test[0, 1:]
    y_qrc = qrc_predictions[0]
    aligned_dates = test_dates[1:]

    har_metrics = evaluate(
        y_true,
        y_har,
        label="HAR only",
    )
    qrc_metrics = evaluate(
        y_true,
        y_qrc,
        label="HAR + QRC",
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    results = pd.DataFrame(
        {
            "date": aligned_dates,
            "ticker": selected_ticker,
            "y_true": y_true,
            "y_har": y_har,
            "y_qrc": y_qrc,
        }
    )

    predictions_path = (
        OUTPUT_DIR / f"qbraid_predictions_{selected_ticker}.csv"
    )
    metrics_path = (
        OUTPUT_DIR / f"qbraid_metrics_{selected_ticker}.json"
    )
    jobs_path = (
        OUTPUT_DIR / f"qbraid_job_ids_{selected_ticker}.json"
    )

    results.to_csv(predictions_path, index=False)

    metrics_path.write_text(
        json.dumps(
            {
                "ticker": selected_ticker,
                "device": DEVICE_ID,
                "num_qubits": NUM_QUBITS,
                "n_trotter": N_TROTTER,
                "feedbacks": FEEDBACKS,
                "shots": SHOTS,
                "train_steps": TRAIN_STEPS,
                "test_steps": TEST_STEPS,
                "tasks_submitted": qrc_model.submitted_tasks,
                "estimated_credits": ESTIMATED_CREDITS,
                "har": har_metrics,
                "har_plus_qrc": qrc_metrics,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    jobs_path.write_text(
        json.dumps(qrc_model.job_ids, indent=2),
        encoding="utf-8",
    )

    print("Subset run completed.")
    print(
        f"Tasks submitted: "
        f"{qrc_model.submitted_tasks}/{REQUIRED_TASKS}"
    )
    print(f"Predictions: {predictions_path}")
    print(f"Metrics:     {metrics_path}")
    print(f"Job IDs:     {jobs_path}")
